# 08 - Agent Workflows

Demonstrates AIMU's agent workflow patterns, aligned with Anthropic's [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) taxonomy.

**Workflows** execute through predetermined code paths: the code controls routing, sequencing, and aggregation; LLMs are invoked at defined points rather than directing their own process.

| Class | Pattern | When to use |
|---|---|---|
| `Router` | Routing | Classify the input, dispatch to a specialist handler |
| `Parallel` | Parallelization | Run multiple agents concurrently, aggregate results |
| `EvaluatorOptimizer` | Evaluator-Optimizer | Generate → evaluate → revise loop |

All three inherit from `Workflow(Runner)`. For sequential chaining and autonomous agents, see **notebook 06**.

## Setup

Create a shared `OllamaClient` with a tool-capable model and attach an `MCPClient` with a few demo tools. The same `model_client` is reused across all sections.

In [ ]:
import datetime
from fastmcp import FastMCP

from aimu.models.ollama import OllamaClient
from aimu.models import StreamPhase
from aimu.tools import MCPClient

mcp = FastMCP("Demo Tools")


@mcp.tool()
def get_current_date_and_time() -> str:
    """Returns the current date and time."""
    return str(datetime.datetime.now())


@mcp.tool()
def get_weather(location: str) -> str:
    """Returns the current weather for a given location."""
    return f"Sunny, 22°C in {location}"  # stubbed


@mcp.tool()
def calculate(expression: str) -> str:
    """Evaluates a simple arithmetic expression and returns the result."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
model_client.mcp_client = MCPClient(server=mcp)

print("Model:", model_client.model.value)
print("Tools:", [t.name for t in model_client.mcp_client.list_tools()])

## A - Router (Routing Pattern)

A `Router` classifies the input with a `routing_agent`, then dispatches the task to the matching handler. Route names are compared case-insensitively.

Handlers can be any `Runner`: a `SimpleAgent`, another `Router`, a `Parallel`, or an `Chain`. An optional `fallback` runner handles unrecognised routes.

Here we set up a router that classifies a query as `weather`, `math`, or `datetime`, and dispatches to a specialist agent with the appropriate system message.

In [ ]:
from aimu.agents import Router, SimpleAgent

router = Router(
    routing_agent=SimpleAgent(
        model_client,
        name="classifier",
        system_message=(
            "Classify the user's query into exactly one of these categories: "
            "weather, math, datetime. "
            "Reply with only the category name; no other text."
        ),
    ),
    handlers={
        "weather": SimpleAgent(
            model_client,
            name="weather-specialist",
            system_message="You are a weather expert. Use the get_weather tool to answer.",
            max_iterations=3,
        ),
        "math": SimpleAgent(
            model_client,
            name="math-specialist",
            system_message="You are a maths expert. Use the calculate tool for all arithmetic.",
            max_iterations=3,
        ),
        "datetime": SimpleAgent(
            model_client,
            name="datetime-specialist",
            system_message="You are a time expert. Use the get_current_date_and_time tool.",
            max_iterations=3,
        ),
    },
    fallback=SimpleAgent(
        model_client,
        name="general",
        system_message="Answer the user's question as helpfully as possible.",
    ),
)

In [ ]:
router.run("What is the weather like in Tokyo?")

In [ ]:
model_client.messages

In [ ]:
router.messages

In [ ]:
result = router.run("What is 17 multiplied by 23?")
result

In [ ]:
model_client.messages

In [ ]:
router.messages

In [ ]:
result = router.run("What time is it right now?")
result

In [ ]:
model_client.messages

### Streaming the Router

`run_streamed()` first yields the classifier's chunks (the route name), then the selected handler's chunks. This lets you see the routing decision in real time.

In [ ]:
model_client.messages = []
current_agent = None

for chunk in router.run_streamed("What is 99 squared?"):
    if chunk.agent_name != current_agent:
        current_agent = chunk.agent_name
        print(f"\n--- [{current_agent}] ---")
    if chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)
    elif chunk.phase == StreamPhase.TOOL_CALLING:
        print(f"[tool: {chunk.content['name']}] → {chunk.content['response']}")

### Router from Config

`Router.from_config()` builds the router from plain dicts; useful for config-driven setups.

In [ ]:
model_client.messages = []

router_from_cfg = Router.from_config(
    routing_config={
        "name": "classifier",
        "system_message": "Classify as one of: weather, math. Reply with only the category name.",
    },
    handler_configs={
        "weather": {"name": "weather-bot", "system_message": "Use the get_weather tool.", "max_iterations": 3},
        "math": {"name": "math-bot", "system_message": "Use the calculate tool.", "max_iterations": 3},
    },
    client=model_client,
)

model_client.messages = []
print(router_from_cfg.run("What is 42 times 7?"))

## B - Parallel (Parallelization Pattern)

A `Parallel` runs multiple worker runners concurrently (via `ThreadPoolExecutor`) and optionally passes their combined output to an aggregator runner.

- Without an aggregator: worker outputs are joined with `separator` and returned directly.
- With an aggregator: the joined outputs become the aggregator's task, which produces the final result.

Workers run to completion before the aggregator starts; this avoids interleaved streaming output.

In [ ]:
from aimu.agents import Parallel

model_client.messages = []

parallel = Parallel(
    workers=[
        SimpleAgent(
            model_client,
            name="optimist",
            system_message="You are an optimist. Give 2-3 benefits of the idea in bullet points.",
        ),
        SimpleAgent(
            model_client,
            name="skeptic",
            system_message="You are a skeptic. Give 2-3 risks or weaknesses of the idea in bullet points.",
        ),
        SimpleAgent(
            model_client,
            name="pragmatist",
            system_message="You are a pragmatist. Give 2-3 practical implementation steps in bullet points.",
        ),
    ],
    aggregator=SimpleAgent(
        model_client,
        name="synthesizer",
        system_message=(
            "You are given three perspectives on an idea: benefits, risks, and implementation steps. "
            "Synthesize them into a balanced 2-paragraph executive summary."
        ),
    ),
    separator="\n\n---\n\n",
)

task = "Build an internal AI assistant for our customer support team."
model_client.messages = []
result = parallel.run(task)
print(result)

### Parallel without an aggregator

Without an aggregator, the worker outputs are joined with `separator` and returned as-is. Useful when you want to collect multiple independent responses and process them yourself.

In [ ]:
model_client.messages = []

perspectives = Parallel(
    workers=[
        SimpleAgent(model_client, name="uk", system_message="Answer from a UK perspective in one sentence."),
        SimpleAgent(model_client, name="us", system_message="Answer from a US perspective in one sentence."),
        SimpleAgent(model_client, name="jp", system_message="Answer from a Japanese perspective in one sentence."),
    ],
    separator="\n",
)

model_client.messages = []
print(perspectives.run("What is the most important meal of the day?"))

### Streaming Parallel output

`run_streamed()` collects all worker results concurrently (not interleaved), then streams the aggregator's response.

In [ ]:
model_client.messages = []

for chunk in parallel.run_streamed("Use AI to improve hospital patient scheduling."):
    if chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## C - EvaluatorOptimizer (Evaluator-Optimizer Pattern)

An `EvaluatorOptimizer` runs a generate → evaluate → revise loop:

1. The `generator` produces an initial response.
2. The `evaluator` receives the task + response and replies with either `PASS` (accepted) or revision feedback.
3. On revision feedback, the generator revises its output.
4. The loop continues until the evaluator returns `PASS` or `max_rounds` is reached.

The key to reliable behaviour is the evaluator's `system_message`; it should say exactly what "good" looks like and use the exact `pass_keyword` string (default `PASS`) when satisfied.

In [ ]:
from aimu.agents import EvaluatorOptimizer

model_client.messages = []

eo = EvaluatorOptimizer(
    generator=SimpleAgent(
        model_client,
        name="writer",
        system_message=(
            "Write a clear, accurate, one-paragraph explanation suitable for a technical audience. "
            "Use precise language."
        ),
    ),
    evaluator=SimpleAgent(
        model_client,
        name="critic",
        system_message=(
            "You are a technical editor. Review the response for accuracy, clarity, and completeness. "
            "If the response is accurate and clearly written, reply with exactly: PASS\n"
            "Otherwise reply with: REVISE: <specific actionable feedback>"
        ),
    ),
    max_rounds=4,
)

model_client.messages = []
result = eo.run("Explain how transformer attention mechanisms work.")
print(result)

### Streaming EvaluatorOptimizer

`run_streamed()` runs all rounds internally and yields the **final accepted output** as a single `GENERATING` chunk; intermediate drafts are not streamed, so the caller only ever sees the accepted result.

In [ ]:
model_client.messages = []

for chunk in eo.run_streamed("Explain the difference between precision and recall."):
    if chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## D - Composing Patterns

All workflow classes accept any `Runner` as their sub-components (agents, workflows, or nested combinations). This enables arbitrarily deep composition.

Here a `Router` dispatches to either:
- A `Parallel` that gathers multiple perspectives (for open-ended questions), or
- An `EvaluatorOptimizer` that produces a polished factual answer (for factual questions).

In [ ]:
model_client.messages = []

# Parallel: gather multiple perspectives on an open-ended question
multi_perspective = Parallel(
    workers=[
        SimpleAgent(model_client, name="angle-1", system_message="Give one key insight on this topic in 2 sentences."),
        SimpleAgent(
            model_client, name="angle-2", system_message="Give one counterargument or limitation in 2 sentences."
        ),
    ],
    aggregator=SimpleAgent(
        model_client,
        name="synthesizer",
        system_message="Combine the two perspectives into a balanced one-paragraph answer.",
    ),
)

# EvaluatorOptimizer: produce a polished factual answer
factual_eo = EvaluatorOptimizer(
    generator=SimpleAgent(
        model_client,
        name="factual-writer",
        system_message="Give a concise, accurate factual answer in one paragraph.",
    ),
    evaluator=SimpleAgent(
        model_client,
        name="fact-checker",
        system_message=(
            "Check if the factual answer is accurate and concise. "
            "If yes, reply: PASS\nIf not, reply: REVISE: <feedback>"
        ),
    ),
    max_rounds=3,
)

# Router dispatches between the two
composed_router = Router(
    routing_agent=SimpleAgent(
        model_client,
        name="type-classifier",
        system_message=("Classify the question as one of: opinion, factual. Reply with only the category name."),
    ),
    handlers={
        "opinion": multi_perspective,
        "factual": factual_eo,
    },
)

for question in [
    "Should companies adopt four-day work weeks?",
    "What is the speed of light in a vacuum?",
]:
    model_client.messages = []
    result = composed_router.run(question)
    print(f"Q: {question}")
    print(f"A: {result}\n{'─' * 60}\n")

## Clean Up

In [ ]:
del model_client, router, router_from_cfg, parallel, perspectives, eo, multi_perspective, factual_eo, composed_router